# 01 · Explore Demonstrations / 理解示教
预计 20 分钟。Observation（观测）是机器人此刻知道的状态；Action（动作）是控制器下一步接收的命令。Demonstration（示教）是一整条人类操作轨迹。按 Shift+Enter 从上到下运行；下方由你填写的答案才是小组提交内容。

In [ ]:
from pathlib import Path
import os, sys, json
from IPython.display import display, Markdown, Image, Video
from manipulation_lab.common import ROOT, DATA, write_json
from manipulation_lab.notebook import workspace, command, next_stage, stage_path
GROUP = os.environ.get("LAB_GROUP", "group_01")  # 教师为每组分配；只改自己的组
RUN = workspace(GROUP, fresh=True)
print("开始一个新实验；重新运行此格会保留旧实验并创建新目录。")

In [ ]:
from manipulation_lab.dataset import inspect
import h5py, numpy as np, matplotlib.pyplot as plt
info = inspect(DATA)
print("Episodes:", info["episode_count"], "Steps:", info["total_steps"])
print("Raw observation shapes:", info["observation_shapes"])
print("Policy input shapes:", info["policy_observation_shapes"])
print("Episode split:", {k: len(v) for k,v in info["splits"].items()})
print(info["time"])
print(info["action_meaning"])

实际策略输入 16 维：末端位置 3、末端四元数 4、夹爪关节 2、Can 世界位姿 7。官方 object 向量共 14 维，本实验只用 object[7:14]，避免相对传感器缓存不一致。位置单位米，世界系 Z 向上，观测四元数 xyzw。动作 7 维：世界系位移、轴角旋转增量和夹爪命令；归一化范围 [-1,1]，正夹爪命令闭合，负值打开。20Hz 是控制频率；没有采集时间戳。

In [ ]:
EPISODE = "demo_0"  # 可选 demo_0 ... demo_199
with h5py.File(DATA, "r") as f:
    d = f["data"][EPISODE]
    expert_actions = d["actions"][:]
    world_position = d["obs/object"][:, 7:10]
    eef_position = d["obs/robot0_eef_pos"][:]
    print("Episode length:", len(expert_actions))
    for step in [0, len(expert_actions)//2, len(expert_actions)-1]:
        print("step", step, "eef", eef_position[step], "object", world_position[step], "action", expert_actions[step])
fig, ax = plt.subplots(2,1,sharex=True,figsize=(8,4))
ax[0].plot(world_position[:,2]); ax[0].set_ylabel("Can world z (m)")
ax[1].plot(expert_actions[:,-1]); ax[1].set_ylabel("Gripper command"); ax[1].set_xlabel("Control step (20Hz)")
plt.show()

In [ ]:
DEMO = next_stage(RUN, "demonstration")
command("replay_demo.py", "--episode", EPISODE, "--output", DEMO)
display(Video(str(DEMO / "demonstration.mp4"), embed=True))

上面是 **人类示教的已记录状态回放**，不是学习策略的闭环推理。下面两题请用自己的语言填写。

In [ ]:
Q1 = ""  # What are the observations and actions in this task?
Q2 = ""  # Why trajectories, instead of only successful final images?
write_json(RUN / "demonstration_analysis.json", {
    "episode": EPISODE, "length": len(expert_actions), "source": "human_demonstration",
    "observations": info["policy_observation_shapes"], "action_dim": 7,
    "q1": Q1, "q2": Q2, "answers_complete": bool(Q1.strip() and Q2.strip()),
    "dataset_sha256": info["dataset_sha256"]})
print("Saved demonstration_analysis.json; 未填写答案时保留草稿标记。")